# Chapter 11: Infrastructure — Deploying the GAS LLM Backend

> **Book:** Agentic AI Engineering  
> **Part:** 2 — RL Methods  
> **System:** Guardian Angel System (GAS) — Multi-agent diabetic patient supervision

In [ ]:
# Install dependencies
# !pip install numpy matplotlib

In [ ]:
import os
TEST_MODE = os.getenv("NOTEBOOK_TEST_MODE", "0") == "1"
print(f"TEST_MODE: {TEST_MODE}")

---
## 🎯 1. Chapter Goal

By the end of this notebook you will:
- Calculate **GPU memory requirements** for deploying LLMs at different precisions
- Understand **continuous batching** and why it dramatically improves throughput
- Simulate **throughput vs batch size** for naive vs continuous batching
- Calculate **GAS deployment requirements** for 5 concurrent agents
- Understand **vLLM configuration** and Docker deployment for the GAS backend

---
## 📖 2. Theory Recap

### LLM Inference Challenges

Serving LLMs efficiently requires solving several problems:

**1. Memory Management**
- Model weights: fixed, loaded once
- KV cache: grows with sequence length, varies per request
- Activations: temporary, per-forward-pass

**2. Batching**
- Naive batching: wait for a full batch, pad to max length → GPU waste
- Continuous batching: add new requests as old ones finish → high utilization

**3. Quantization**
- fp32: 4 bytes/param (training)
- bf16/fp16: 2 bytes/param (standard inference)
- int8: 1 byte/param (quantized)
- int4: 0.5 bytes/param (aggressive quantization)

### vLLM: PagedAttention

vLLM introduces **PagedAttention** — managing KV cache like OS virtual memory:
- KV cache stored in non-contiguous memory blocks
- Blocks allocated/freed dynamically
- Enables continuous batching with near-zero memory waste
- 2-24x throughput improvement over naive serving

### Key Metrics

| Metric | Definition |
|--------|------------|
| TTFT | Time to First Token |
| TPOT | Time Per Output Token |
| Throughput | Tokens/second across all requests |
| Latency | End-to-end response time |

---
## 🔨 3. Build It

### 3.1 Memory Calculator

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def estimate_model_memory(
    params_b: float,
    precision: str = 'bf16',
    batch_size: int = 1,
    max_seq_len: int = 2048,
    num_layers: int = None
) -> dict:
    """
    Estimate GPU memory requirements for LLM inference.
    
    Args:
        params_b: model size in billions of parameters
        precision: 'fp32', 'bf16', 'fp16', 'int8', or 'int4'
        batch_size: number of concurrent requests
        max_seq_len: maximum sequence length
        num_layers: number of transformer layers (estimated if None)
    
    Returns:
        dict with memory breakdown in GB
    """
    bytes_per_param = {
        'fp32': 4,
        'bf16': 2,
        'fp16': 2,
        'int8': 1,
        'int4': 0.5
    }
    
    if precision not in bytes_per_param:
        raise ValueError(f"Unknown precision: {precision}. Choose from {list(bytes_per_param.keys())}")
    
    bpp = bytes_per_param[precision]
    
    # Model weights
    weights_gb = params_b * 1e9 * bpp / 1e9
    
    # KV cache estimate
    # Rule of thumb: ~0.125 GB per billion params per batch item at 2048 tokens
    kv_cache_gb = params_b * 0.125 * batch_size * (max_seq_len / 2048)
    
    # Activation memory (rough estimate)
    activation_gb = params_b * 0.05 * batch_size
    
    total_gb = weights_gb + kv_cache_gb + activation_gb
    
    return {
        'weights_gb': round(weights_gb, 2),
        'kv_cache_gb': round(kv_cache_gb, 2),
        'activation_gb': round(activation_gb, 2),
        'total_gb': round(total_gb, 2),
        'precision': precision,
        'params_b': params_b,
        'batch_size': batch_size
    }

# Calculate memory for common model sizes
model_sizes = [3, 7, 13, 70]
precisions = ['fp32', 'bf16', 'int8', 'int4']

print("GPU Memory Requirements (GB) — Inference, batch_size=1")
print("=" * 65)
print(f"{'Model':>10} {'fp32':>8} {'bf16':>8} {'int8':>8} {'int4':>8}")
print("-" * 65)
for size in model_sizes:
    row = f"{size}B params"
    for prec in precisions:
        mem = estimate_model_memory(size, prec)
        row += f"  {mem['total_gb']:>6.1f}"
    print(row)

print()
# Detailed breakdown for 7B model
print("Detailed breakdown: 7B model, bf16, batch_size=1")
mem_7b = estimate_model_memory(7, 'bf16', batch_size=1)
for k, v in mem_7b.items():
    if k.endswith('_gb'):
        print(f"  {k:<20}: {v:>6.2f} GB")
print(f"  {'TOTAL':<20}: {mem_7b['total_gb']:>6.2f} GB")

### 3.2 Throughput Simulation: Naive vs Continuous Batching

In [ ]:
def simulate_naive_batching(
    batch_sizes: list,
    base_tokens_per_sec: float = 100.0,
    avg_seq_len: int = 256,
    padding_overhead: float = 0.3
) -> list:
    """
    Simulate throughput for naive batching.
    
    Naive batching:
    - Waits for full batch before processing
    - Pads all sequences to max length in batch
    - GPU utilization drops as batch size increases (more padding)
    """
    throughputs = []
    for bs in batch_sizes:
        # Throughput increases with batch size but padding overhead grows
        # Padding overhead: longer wait + more padding tokens
        effective_utilization = 1.0 - padding_overhead * np.log(bs + 1) / np.log(32)
        effective_utilization = max(0.3, effective_utilization)
        
        # Throughput = batch_size * tokens_per_seq / time_per_batch
        # Time per batch grows sub-linearly with batch size
        time_per_batch = avg_seq_len / (base_tokens_per_sec * effective_utilization)
        throughput = bs * avg_seq_len / time_per_batch
        throughputs.append(throughput)
    return throughputs

def simulate_continuous_batching(
    batch_sizes: list,
    base_tokens_per_sec: float = 100.0,
    avg_seq_len: int = 256
) -> list:
    """
    Simulate throughput for continuous batching (vLLM-style).
    
    Continuous batching:
    - Processes tokens as they're generated
    - New requests join as old ones finish
    - Near-constant GPU utilization
    - No padding waste
    """
    throughputs = []
    for bs in batch_sizes:
        # Near-linear scaling with batch size (high GPU utilization)
        # Slight overhead for request management
        utilization = 0.92 - 0.002 * bs  # Slight decrease at very high batch sizes
        utilization = max(0.75, utilization)
        
        throughput = bs * base_tokens_per_sec * utilization
        throughputs.append(throughput)
    return throughputs

batch_sizes = [1, 2, 4, 8, 16, 32, 64]
naive_throughput = simulate_naive_batching(batch_sizes)
continuous_throughput = simulate_continuous_batching(batch_sizes)

print("Throughput Comparison: Naive vs Continuous Batching")
print("=" * 55)
print(f"{'Batch Size':>12} {'Naive (tok/s)':>15} {'Continuous (tok/s)':>20} {'Speedup':>10}")
print("-" * 55)
for bs, naive, cont in zip(batch_sizes, naive_throughput, continuous_throughput):
    speedup = cont / naive
    print(f"{bs:>12} {naive:>15.1f} {cont:>20.1f} {speedup:>9.1f}x")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(batch_sizes, naive_throughput, 'ro-', linewidth=2, markersize=8, label='Naive batching')
ax1.plot(batch_sizes, continuous_throughput, 'go-', linewidth=2, markersize=8, label='Continuous batching (vLLM)')
ax1.set_title('Throughput vs Batch Size', fontsize=12, fontweight='bold')
ax1.set_xlabel('Batch Size (concurrent requests)')
ax1.set_ylabel('Throughput (tokens/second)')
ax1.legend()
ax1.grid(alpha=0.3)
ax1.set_xscale('log', base=2)

speedups = [c/n for c, n in zip(continuous_throughput, naive_throughput)]
ax2.bar([str(bs) for bs in batch_sizes], speedups, color='#3498db', edgecolor='black')
ax2.axhline(y=1.0, color='red', linestyle='--', linewidth=1.5, label='No speedup')
ax2.set_title('Continuous Batching Speedup over Naive', fontsize=12, fontweight='bold')
ax2.set_xlabel('Batch Size')
ax2.set_ylabel('Speedup (x)')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('LLM Serving: Naive vs Continuous Batching', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('batching_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

### 3.3 GAS Deployment Calculator

In [ ]:
def gas_deployment_calculator(
    model_size_b: float,
    precision: str,
    n_agents: int = 5,
    concurrent_requests_per_agent: int = 3,
    shared_model: bool = True
) -> dict:
    """
    Calculate GPU requirements for the GAS multi-agent system.
    
    Args:
        model_size_b: model size in billions
        precision: inference precision
        n_agents: number of GAS agents (default: 5)
        concurrent_requests_per_agent: requests each agent handles simultaneously
        shared_model: if True, all agents share one model instance (vLLM)
    """
    total_concurrent = n_agents * concurrent_requests_per_agent
    
    if shared_model:
        # vLLM: one model instance serves all agents
        mem = estimate_model_memory(model_size_b, precision, batch_size=total_concurrent)
        total_memory = mem['total_gb']
        model_copies = 1
    else:
        # Naive: each agent has its own model instance
        mem = estimate_model_memory(model_size_b, precision, batch_size=concurrent_requests_per_agent)
        total_memory = mem['total_gb'] * n_agents
        model_copies = n_agents
    
    # GPU selection
    gpu_options = [
        {'name': 'A100 40GB', 'vram': 40},
        {'name': 'A100 80GB', 'vram': 80},
        {'name': 'H100 80GB', 'vram': 80},
        {'name': '4x A100 40GB', 'vram': 160},
        {'name': '2x H100 80GB', 'vram': 160},
    ]
    
    suitable_gpus = [g for g in gpu_options if g['vram'] >= total_memory * 1.1]  # 10% headroom
    
    return {
        'model_size_b': model_size_b,
        'precision': precision,
        'n_agents': n_agents,
        'concurrent_requests_per_agent': concurrent_requests_per_agent,
        'total_concurrent_requests': total_concurrent,
        'shared_model': shared_model,
        'model_copies': model_copies,
        'total_memory_gb': round(total_memory, 1),
        'memory_breakdown': mem,
        'suitable_gpus': suitable_gpus
    }

# GAS deployment scenarios
scenarios = [
    {'model': 7, 'precision': 'bf16', 'shared': True, 'label': '7B bf16 (shared)'},
    {'model': 7, 'precision': 'int4', 'shared': True, 'label': '7B int4 (shared)'},
    {'model': 13, 'precision': 'bf16', 'shared': True, 'label': '13B bf16 (shared)'},
    {'model': 7, 'precision': 'bf16', 'shared': False, 'label': '7B bf16 (per-agent)'},
]

print("GAS Deployment Calculator: 5 Agents × 3 Concurrent Requests")
print("=" * 70)

for s in scenarios:
    result = gas_deployment_calculator(
        s['model'], s['precision'], 
        n_agents=5, concurrent_requests_per_agent=3,
        shared_model=s['shared']
    )
    print(f"\nScenario: {s['label']}")
    print(f"  Total concurrent requests: {result['total_concurrent_requests']}")
    print(f"  Model copies:              {result['model_copies']}")
    print(f"  Total GPU memory needed:   {result['total_memory_gb']} GB")
    if result['suitable_gpus']:
        print(f"  Suitable GPUs:             {', '.join(g['name'] for g in result['suitable_gpus'][:2])}")
    else:
        print(f"  Suitable GPUs:             ⚠️  Requires multi-GPU setup")

### 3.4 vLLM Configuration for GAS

In [ ]:
# vLLM configuration for GAS LLM backend
# This shows the configuration — execution requires GPU and model download

vllm_config = '''
# vLLM server configuration for GAS LLM backend
# Launch with: python -m vllm.entrypoints.openai.api_server --config gas_vllm_config.yaml

from vllm import LLM, SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs

# Engine configuration
engine_args = AsyncEngineArgs(
    model="./gas_advisor_dpo",          # Path to trained GAS Advisor
    tokenizer="./gas_advisor_dpo",
    
    # Memory
    dtype="bfloat16",
    gpu_memory_utilization=0.90,        # Use 90% of GPU memory
    max_model_len=4096,                 # Max sequence length
    
    # Batching
    max_num_seqs=15,                    # 5 agents × 3 concurrent = 15
    max_num_batched_tokens=8192,
    
    # Performance
    enable_prefix_caching=True,         # Cache common prefixes (system prompt)
    use_v2_block_manager=True,          # PagedAttention v2
    
    # Quantization (optional, for memory savings)
    # quantization="awq",               # Uncomment for 4-bit quantization
)

# Sampling parameters for GAS agents
gas_sampling_params = SamplingParams(
    temperature=0.7,        # Some creativity for advice generation
    top_p=0.9,
    max_tokens=512,         # Medical advice should be concise
    stop=["</s>", "[INST]"],  # Stop at end of response
)

# Safety Critic uses lower temperature (more deterministic)
safety_critic_params = SamplingParams(
    temperature=0.1,        # Near-deterministic for safety scoring
    max_tokens=64,          # Just needs a score/verdict
)
'''

if not TEST_MODE:
    print("vLLM Configuration for GAS:")
    print(vllm_config)
else:
    print("TEST_MODE: vLLM config shown (not executing)")

print("\nKey vLLM settings for GAS:")
print("  max_num_seqs=15          → 5 agents × 3 concurrent requests")
print("  enable_prefix_caching    → cache system prompt (saves ~20% memory)")
print("  use_v2_block_manager     → PagedAttention v2 for better KV cache")
print("  gpu_memory_utilization   → leave 10% headroom for safety")

### 3.5 Docker Deployment Configuration

In [ ]:
dockerfile_content = '''
# Dockerfile for GAS LLM Backend
# Build: docker build -t gas-llm-backend .
# Run:   docker run --gpus all -p 8000:8000 gas-llm-backend

FROM vllm/vllm-openai:latest

# Install additional dependencies
RUN pip install sentence-transformers==2.7.0

# Copy model and configuration
COPY ./gas_advisor_dpo /models/gas_advisor
COPY ./gas_vllm_config.yaml /app/config.yaml

# Health check
HEALTHCHECK --interval=30s --timeout=10s --start-period=60s \\
    CMD curl -f http://localhost:8000/health || exit 1

# Expose OpenAI-compatible API
EXPOSE 8000

# Launch vLLM server
CMD ["python", "-m", "vllm.entrypoints.openai.api_server", \\
     "--model", "/models/gas_advisor", \\
     "--host", "0.0.0.0", \\
     "--port", "8000", \\
     "--max-num-seqs", "15", \\
     "--gpu-memory-utilization", "0.90", \\
     "--enable-prefix-caching", \\
     "--dtype", "bfloat16"]
'''

docker_compose = '''
# docker-compose.yml for full GAS system
version: '3.8'

services:
  # LLM Backend (vLLM)
  gas-llm-backend:
    image: gas-llm-backend:latest
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia
              count: 1
              capabilities: [gpu]
    ports:
      - "8000:8000"
    volumes:
      - ./models:/models
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3

  # GAS Orchestrator
  gas-orchestrator:
    image: gas-orchestrator:latest
    environment:
      - LLM_BACKEND_URL=http://gas-llm-backend:8000
      - N_AGENTS=5
    depends_on:
      gas-llm-backend:
        condition: service_healthy
    ports:
      - "8080:8080"

  # Redis for agent state
  redis:
    image: redis:7-alpine
    ports:
      - "6379:6379"
'''

print("Dockerfile for GAS LLM Backend:")
print(dockerfile_content)
print("\ndocker-compose.yml for Full GAS System:")
print(docker_compose)

### 3.6 Memory Visualization: GAS Deployment Options

In [ ]:
# Visualize memory requirements for different GAS deployment options
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Memory by model size and precision
ax1 = axes[0]
model_sizes_plot = [3, 7, 13, 70]
precisions_plot = ['fp32', 'bf16', 'int8', 'int4']
colors_prec = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

x = np.arange(len(model_sizes_plot))
width = 0.2

for i, (prec, color) in enumerate(zip(precisions_plot, colors_prec)):
    memories = [estimate_model_memory(s, prec)['weights_gb'] for s in model_sizes_plot]
    ax1.bar(x + i*width, memories, width, label=prec, color=color, edgecolor='black', linewidth=0.5)

# GPU memory lines
ax1.axhline(y=40, color='purple', linestyle='--', alpha=0.7, linewidth=1.5, label='A100 40GB')
ax1.axhline(y=80, color='navy', linestyle='--', alpha=0.7, linewidth=1.5, label='A100/H100 80GB')

ax1.set_title('Model Weights Memory by Size & Precision', fontsize=12, fontweight='bold')
ax1.set_xlabel('Model Size')
ax1.set_ylabel('Memory (GB)')
ax1.set_xticks(x + width * 1.5)
ax1.set_xticklabels([f'{s}B' for s in model_sizes_plot])
ax1.legend(fontsize=8)
ax1.grid(axis='y', alpha=0.3)

# Plot 2: GAS total memory (5 agents, shared vs per-agent)
ax2 = axes[1]
scenarios_plot = [
    ('7B\nbf16\nshared', gas_deployment_calculator(7, 'bf16', shared_model=True)['total_memory_gb']),
    ('7B\nint4\nshared', gas_deployment_calculator(7, 'int4', shared_model=True)['total_memory_gb']),
    ('13B\nbf16\nshared', gas_deployment_calculator(13, 'bf16', shared_model=True)['total_memory_gb']),
    ('7B\nbf16\nper-agent', gas_deployment_calculator(7, 'bf16', shared_model=False)['total_memory_gb']),
]

labels_s = [s[0] for s in scenarios_plot]
memories_s = [s[1] for s in scenarios_plot]
bar_colors = ['#2ecc71' if m <= 80 else ('#f39c12' if m <= 160 else '#e74c3c') for m in memories_s]

bars = ax2.bar(labels_s, memories_s, color=bar_colors, edgecolor='black')
ax2.axhline(y=40, color='purple', linestyle='--', alpha=0.7, linewidth=1.5, label='A100 40GB')
ax2.axhline(y=80, color='navy', linestyle='--', alpha=0.7, linewidth=1.5, label='A100/H100 80GB')
ax2.axhline(y=160, color='red', linestyle='--', alpha=0.7, linewidth=1.5, label='4×A100 40GB')

for bar, mem in zip(bars, memories_s):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
             f'{mem:.0f}GB', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax2.set_title('GAS Total Memory: 5 Agents × 3 Concurrent Requests', fontsize=12, fontweight='bold')
ax2.set_ylabel('Total GPU Memory (GB)')
ax2.legend(fontsize=8)
ax2.grid(axis='y', alpha=0.3)

green_patch = mpatches.Patch(color='#2ecc71', label='Fits on 1 GPU')
orange_patch = mpatches.Patch(color='#f39c12', label='Needs 2 GPUs')
red_patch = mpatches.Patch(color='#e74c3c', label='Needs 4+ GPUs')
ax2.legend(handles=[green_patch, orange_patch, red_patch] + ax2.get_legend_handles_labels()[0][3:], fontsize=8)

plt.suptitle('GAS Infrastructure: Memory Planning', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('gas_memory_planning.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nRecommendation for GAS:")
print("  ✅ 7B int4 shared (vLLM): fits on A100 40GB, good quality")
print("  ✅ 7B bf16 shared (vLLM): fits on A100 80GB, best quality")
print("  ❌ Per-agent deployment: 5x memory overhead, not recommended")

---
## 🏥 4. GAS Connection

### Deploying the Guardian Angel System

The GAS system needs to serve **5 agents concurrently** with low latency for real-time patient monitoring.

```
GAS Production Architecture:

┌─────────────────────────────────────────────────────┐
│                  GAS Orchestrator                   │
│  ┌──────────┐ ┌──────────┐ ┌──────────┐             │
│  │ Monitor  │ │ Advisor  │ │ Safety   │ ...5 agents │
│  │  Agent   │ │  Agent   │ │ Critic   │             │
│  └────┬─────┘ └────┬─────┘ └────┬─────┘             │
│       └────────────┴────────────┘                   │
│                    │ OpenAI-compatible API          │
└────────────────────┼────────────────────────────────┘
                     ↓
┌─────────────────────────────────────────────────────┐
│              vLLM LLM Backend                       │
│  - GAS Advisor model (DPO-aligned)                  │
│  - PagedAttention KV cache                          │
│  - Continuous batching (15 concurrent requests)     │
│  - Prefix caching (system prompt)                   │
│  GPU: A100 80GB or H100 80GB                        │
└─────────────────────────────────────────────────────┘
```

**The GAS system needs to serve 5 agents concurrently.**

Key infrastructure decisions:
1. **Shared model** (vLLM): All 5 agents share one model instance — 5x memory savings vs per-agent
2. **Continuous batching**: Handles bursty traffic from multiple agents without queuing
3. **Prefix caching**: The medical system prompt is the same for all agents — cache it
4. **int4 quantization**: If memory-constrained, 4-bit quantization with minimal quality loss
5. **Health checks**: Medical system must have automatic failover and health monitoring

---
## ✅ 5. Checkpoint

Test your understanding:

**Q1: How much memory does a 7B model require in bf16 precision?**  
A: 7B × 2 bytes/param = 14 GB for weights alone. With KV cache and activations for a single request, approximately 15-16 GB total. For 15 concurrent requests (5 agents × 3), approximately 28-30 GB.

**Q2: What is the key advantage of continuous batching over naive batching?**  
A: Continuous batching processes tokens as they're generated and adds new requests as old ones finish, maintaining near-constant GPU utilization. Naive batching waits for a full batch and pads all sequences to max length, wasting GPU cycles on padding tokens.

**Q3: Why does vLLM use PagedAttention?**  
A: PagedAttention manages the KV cache like OS virtual memory — storing it in non-contiguous blocks that can be allocated and freed dynamically. This eliminates memory fragmentation, allows more requests to be served simultaneously, and enables efficient memory sharing for beam search.

**Q4: Why is prefix caching particularly valuable for GAS?**  
A: All 5 GAS agents use the same medical system prompt (patient context, safety guidelines). Prefix caching stores the KV cache for this common prefix, so it only needs to be computed once — saving significant computation for every request.

**Q5: What is the recommended GAS deployment configuration and why?**  
A: 7B model in bf16 with vLLM shared serving on an A100 80GB. This provides: (1) good model quality, (2) fits in 80GB with 15 concurrent requests, (3) shared model saves 5x memory vs per-agent, (4) vLLM's continuous batching handles bursty multi-agent traffic efficiently.